In [1]:
from google.colab import files
uploaded = files.upload()

Saving Housing.csv to Housing.csv


In [5]:

# ===============================
# IMPORT LIBRARIES
# ===============================
import numpy as np
import pandas as pd

# ===============================
# LOAD DATASET
# ===============================
df = pd.read_csv("Housing.csv")   # change name if needed

print("Dataset Preview:")
print(df.head())

# ===============================
# FIX CATEGORICAL DATA
# ===============================

# yes/no → 1/0
binary_cols = [
    'mainroad', 'guestroom', 'basement',
    'hotwaterheating', 'airconditioning', 'prefarea'
]

for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# furnishingstatus → numeric
df['furnishingstatus'] = df['furnishingstatus'].map({
    'furnished': 2,
    'semi-furnished': 1,
    'unfurnished': 0
})

print("\nAfter Encoding:")
print(df.head())

# ===============================
# PREPARE DATA (IMPORTANT FIX)
# ===============================
# Target = price
y = df['price'].values

# Features = everything except price
X = df.drop('price', axis=1).values

# ===============================
# NORMALIZATION
# ===============================
def normalize(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

X = normalize(X)

# ===============================
# TRAIN TEST SPLIT
# ===============================
def train_test_split(X, y, test_size=0.2):
    indices = np.arange(len(X))
    np.random.shuffle(indices)

    split = int(len(X) * (1 - test_size))

    return X[indices[:split]], X[indices[split:]], y[indices[:split]], y[indices[split:]]

X_train, X_test, y_train, y_test = train_test_split(X, y)

# ===============================
# ADD BIAS TERM
# ===============================
def add_bias(X):
    return np.c_[np.ones(X.shape[0]), X]

X_train = add_bias(X_train)
X_test = add_bias(X_test)

# ===============================
# TRAIN MODEL (GRADIENT DESCENT)
# ===============================
def train_linear_regression(X, y, lr=0.01, epochs=1000):
    m, n = X.shape
    weights = np.zeros(n)

    for _ in range(epochs):
        y_pred = np.dot(X, weights)
        error = y_pred - y

        gradient = (1/m) * np.dot(X.T, error)
        weights -= lr * gradient

    return weights

weights = train_linear_regression(X_train, y_train)

# ===============================
# PREDICTION
# ===============================
def predict(X, weights):
    return np.dot(X, weights)

y_pred = predict(X_test, weights)

# ===============================
# EVALUATION
# ===============================
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def r2_score(y_true, y_pred):
    ss_total = np.sum((y_true - np.mean(y_true))**2)
    ss_residual = np.sum((y_true - y_pred)**2)
    return 1 - (ss_residual / ss_total)

print("\nFinal Results:")
print("Weights:", weights)
print("MSE:", mse(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

Dataset Preview:
      price  area  bedrooms  bathrooms  stories mainroad guestroom basement  \
0  13300000  7420         4          2        3      yes        no       no   
1  12250000  8960         4          4        4      yes        no       no   
2  12250000  9960         3          2        2      yes        no      yes   
3  12215000  7500         4          2        2      yes        no      yes   
4  11410000  7420         4          1        2      yes       yes      yes   

  hotwaterheating airconditioning  parking prefarea furnishingstatus  
0              no             yes        2      yes        furnished  
1              no             yes        3       no        furnished  
2              no              no        2      yes   semi-furnished  
3              no             yes        3      yes        furnished  
4              no             yes        2       no        furnished  

After Encoding:
      price  area  bedrooms  bathrooms  stories  mainroad  guestr